## Build an Image Classifier

Use FashionMNIST dataset, load it via TorchVision. Split into train/test/validation.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms.v2 as T
import torchmetrics

device = 'mps'

In [3]:
# preprocessing pipeline w/ 2 chained transforms
# first converts raw image format (PIL image) to TorchVisions Image type (subclass of tensor)
# second transform converts datatype to 32bit floats, and scale=True scales to 0.0-1.0 (better for nn)
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

# root arg is the path to dir where torchvision creates subdir for fashion mnist dataset
# download arg says if we want to download if it cant be found localy
# transform= uses our custom pipeline
train_and_valid_data = torchvision.datasets.FashionMNIST( # loads the training portion, transforms via our pipeline
    root="datasets", train=True, download=True, transform=toTensor)
test_data = torchvision.datasets.FashionMNIST( # load test
    root='datasets', train=False, download=True, transform=toTensor)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split( # randomly split 60k train to 55k train 5k validation
    train_and_valid_data, [55_000, 5_000]
)

In [4]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

In [5]:
X_sample, y_sample = train_data[0]
# dim1 is channel dimension, js 1 for b/w. higher = darker
# dim2 and 3 are height and width
X_sample.shape

torch.Size([1, 28, 28])

In [6]:
X_sample.dtype # contains torch floats

torch.float32

In [7]:
# target is int 0-9, rep. idx in list of class names
print(y_sample)
# dataset has classes attib w/ list of class names
train_and_valid_data.classes[y_sample]

9


'Ankle boot'

 now we can build the classifier. 

 we will build a custom module for a classification mlp w/ two hidden layers. don't need a custom module, but its cleaner and easier to tune the hyperparams
 - in this case we can specifiy the # of neurons in hidden layers to tune the model without rewriting it since it's not hardcoded
 - also change # of inputs/outputs if we wanna use w/ different data

 ctor params rep. # inputs, neurons in hidden layer, and # of classes for outputs (1 output per class)

 note that nn.flatten() reshapes each input sample to 1d, needed for nn.Linear layers 
 for a batch of size 32 that's [32, 1, 28, 28] -> [32, 784] so the matmul shape lines up correct (weights are a vector w/ 784 entries)

In [8]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential( # mlp is just sequential linear 
            nn.Flatten(), # reshapes each input sample to 1d, needed for nn.Linear layers for batch: [32, 1, 28, 28]
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes)
        )

    def forward(self, X):
        return self.mlp(X)

In [9]:
torch.manual_seed(42)
model = ImageClassifier(n_inputs=28*28, n_hidden1=300, n_hidden2=100, n_classes=10)
xentropy = nn.CrossEntropyLoss() # since mutliclass classification, CEloss accepts class indicies or class prob

note that xentropy calcs loss straight from the class scores (logits) not from the class prob, internally applying softmax. faster computation, and more numerically stable (float point errors)

but model must output logits, so we need to call softmax ourselves if we want prob

In [10]:
def evaluate_tm(model, data_loader, metric):
    model.eval() # eval mode
    metric.reset() # clear any state from the last time we used the metric
    with torch.no_grad(): # no computation graph needed
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device) # copy to gpu
            y_pred = model(X_batch) # logits, shape [32, 10]
            metric.update(y_pred, y_batch) # accuracy takes the argmax of the logits for us

    return metric.compute() # accuracy over the whole dataset

In [11]:
def train(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs):
    for epoch in range(n_epochs):
        model.train() # set back to training mode each epoch, since evaluate_tm switches to eval mode
        total_loss = 0.
        for X_batch, y_batch in train_loader: # X_batch: [32, 1, 28, 28], y_batch: [32] class indices
            X_batch, y_batch = X_batch.to(device), y_batch.to(device) # copy batches to gpu
            y_pred = model(X_batch) # forward pass, logits [32, 10]
            loss = criterion(y_pred, y_batch) # cross entropy from logits + class indices
            total_loss += loss.item()
            loss.backward() # compute gradients
            optimizer.step() # update weights and biases
            optimizer.zero_grad() # clear gradients for next batch

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")
        print(f"Training accuracy: {evaluate_tm(model, train_loader, metric):.4f}")
        print(f"Validation accuracy: {evaluate_tm(model, valid_loader, metric):.4f}")

In [12]:
model = model.to(device) # move to gpu
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # stochastic gradient descent
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device) # accuracy over 10 classes, move to gpu
train(model, optimizer, xentropy, accuracy, train_loader, valid_loader, n_epochs=20)

Epoch 1/20, Loss: 1.1357
Training accuracy: 0.7533
Validation accuracy: 0.7446
Epoch 2/20, Loss: 0.6019
Training accuracy: 0.8136
Validation accuracy: 0.8036
Epoch 3/20, Loss: 0.5135
Training accuracy: 0.8243
Validation accuracy: 0.8132
Epoch 4/20, Loss: 0.4742
Training accuracy: 0.8348
Validation accuracy: 0.8204
Epoch 5/20, Loss: 0.4491
Training accuracy: 0.8450
Validation accuracy: 0.8366
Epoch 6/20, Loss: 0.4298
Training accuracy: 0.8512
Validation accuracy: 0.8428
Epoch 7/20, Loss: 0.4147
Training accuracy: 0.8585
Validation accuracy: 0.8496
Epoch 8/20, Loss: 0.4008
Training accuracy: 0.8641
Validation accuracy: 0.8526
Epoch 9/20, Loss: 0.3876
Training accuracy: 0.8696
Validation accuracy: 0.8558
Epoch 10/20, Loss: 0.3775
Training accuracy: 0.8645
Validation accuracy: 0.8494
Epoch 11/20, Loss: 0.3665
Training accuracy: 0.8746
Validation accuracy: 0.8608
Epoch 12/20, Loss: 0.3584
Training accuracy: 0.8771
Validation accuracy: 0.8646
Epoch 13/20, Loss: 0.3499
Training accuracy: 0.88

make preds on new images

In [14]:
model.eval()
X_new, y_new = next(iter(valid_loader)) # create iter over loader, next pulls 1 batch (first), assign it
X_new = X_new[:3].to(device) # move first 3 images to gpu
with torch.no_grad(): # forward pass on first 3, aka make preds 
    y_pred_logits = model(X_new) # gives us the raw scores per class for each instance

y_pred = y_pred_logits.argmax(dim=1) # idx of largest logit (highest score) is our concrete prediction
y_pred

tensor([7, 4, 2], device='mps:0')

In [ ]:
[train_and_valid_data.classes[index] for index in y_pred] # get class names based on indexes (all correct in this case)

['Sneaker', 'Coat', 'Pullover']

if we want models estimated probs, we need to calc softmax ourselves since we didnt add softmax acitavtion func on output layer

In [ ]:
import torch.nn.functional as F

# dim=1 means apply along class dim -> normalize across classes in 1 row (each pred)
y_proba = F.softmax(y_pred_logits, dim=1) 
y_proba.round(decimals=3)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0090, 0.0000, 0.9680, 0.0030,
         0.0190],
        [0.0000, 0.0000, 0.0180, 0.0000, 0.9810, 0.0000, 0.0010, 0.0000, 0.0000,
         0.0000],
        [0.0010, 0.0000, 0.8370, 0.0040, 0.1010, 0.0000, 0.0530, 0.0000, 0.0040,
         0.0000]], device='mps:0')

get top k (4) probs 

In [17]:
y_top4_logits, y_top4_indices = torch.topk(y_pred_logits, k=4, dim=1)
y_top4_probas = F.softmax(y_top4_logits, dim=1)
y_top4_probas.round(decimals=3)

tensor([[0.9680, 0.0190, 0.0090, 0.0030],
        [0.9810, 0.0180, 0.0010, 0.0000],
        [0.8410, 0.1020, 0.0530, 0.0040]], device='mps:0')

In [18]:
y_top4_indices

tensor([[7, 9, 5, 8],
        [4, 2, 6, 8],
        [2, 4, 6, 3]], device='mps:0')

## Fine Tuning NN Hyperparams via Optuna

to autosearch for good hypeparams we can conver pytorch model to sklearn estimator
- write own custom estimator class
- use wrapper library

then use gs cv or rs cv to fine tune 

or we can use fine tuning library e.g. optuna
- optuna will repeatedly call objective and try and maximize its result

In [19]:
import optuna

def objective(trial):
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True) # sample lr from log dist 
    n_hidden = trial.suggest_int('n_hidden', 20, 300) # sample # of neurons for hidden layer from uniform dist

    # build fresh model for this trial using suggested hidden value for both layers
    model = ImageClassifier(n_inputs=1*28*28, n_hidden1=n_hidden, n_hidden2=n_hidden, n_classes=10).to(device) 
    # new optimizer using suggested lr 
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    # train, eval, return
    n_epochs = 20
    train(model, optimizer, xentropy, accuracy, train_loader, valid_loader, n_epochs)
    validation_accuracy = evaluate_tm(model, valid_loader, accuracy)

    # return score for this trial. optuna tries to maximize the score
    return validation_accuracy

/Users/zainbabar/code/intro-pytorch/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42) # algo optuna uses to decide which hyperparm values to try
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=5) # runs obj function 5 times, keeps the model that did the best

[I 2026-09-26 18:39:25,449] A new study created in memory with name: no-name-20a85cc3-75e1-463a-8b0c-c8405bfb15b4


Epoch 1/20, Loss: 2.2769
Training accuracy: 0.1904
Validation accuracy: 0.1862
Epoch 2/20, Loss: 2.2093
Training accuracy: 0.3576
Validation accuracy: 0.3488
Epoch 3/20, Loss: 2.1164
Training accuracy: 0.4631
Validation accuracy: 0.4564
Epoch 4/20, Loss: 1.9776
Training accuracy: 0.5553
Validation accuracy: 0.5558
Epoch 5/20, Loss: 1.7867
Training accuracy: 0.6082
Validation accuracy: 0.6072
Epoch 6/20, Loss: 1.5774
Training accuracy: 0.6270
Validation accuracy: 0.6214
Epoch 7/20, Loss: 1.3979
Training accuracy: 0.6304
Validation accuracy: 0.6254
Epoch 8/20, Loss: 1.2605
Training accuracy: 0.6415
Validation accuracy: 0.6322
Epoch 9/20, Loss: 1.1572
Training accuracy: 0.6465
Validation accuracy: 0.6340
Epoch 10/20, Loss: 1.0782
Training accuracy: 0.6604
Validation accuracy: 0.6484
Epoch 11/20, Loss: 1.0162
Training accuracy: 0.6669
Validation accuracy: 0.6560
Epoch 12/20, Loss: 0.9665
Training accuracy: 0.6726
Validation accuracy: 0.6628
Epoch 13/20, Loss: 0.9258
Training accuracy: 0.68

[I 2026-09-26 18:42:01,135] Trial 0 finished with value: 0.7093999981880188 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.7093999981880188.


Epoch 1/20, Loss: 1.1463
Training accuracy: 0.7378
Validation accuracy: 0.7262
Epoch 2/20, Loss: 0.6166
Training accuracy: 0.8016
Validation accuracy: 0.7938
Epoch 3/20, Loss: 0.5204
Training accuracy: 0.8272
Validation accuracy: 0.8176
Epoch 4/20, Loss: 0.4787
Training accuracy: 0.8420
Validation accuracy: 0.8268
Epoch 5/20, Loss: 0.4541
Training accuracy: 0.8384
Validation accuracy: 0.8238
Epoch 6/20, Loss: 0.4366
Training accuracy: 0.8490
Validation accuracy: 0.8350
Epoch 7/20, Loss: 0.4211
Training accuracy: 0.8578
Validation accuracy: 0.8426
Epoch 8/20, Loss: 0.4093
Training accuracy: 0.8643
Validation accuracy: 0.8510
Epoch 9/20, Loss: 0.3976
Training accuracy: 0.8682
Validation accuracy: 0.8524
Epoch 10/20, Loss: 0.3875
Training accuracy: 0.8694
Validation accuracy: 0.8548
Epoch 11/20, Loss: 0.3788
Training accuracy: 0.8723
Validation accuracy: 0.8562
Epoch 12/20, Loss: 0.3704
Training accuracy: 0.8759
Validation accuracy: 0.8598
Epoch 13/20, Loss: 0.3628
Training accuracy: 0.87

[I 2026-09-26 18:44:27,940] Trial 1 finished with value: 0.8748000264167786 and parameters: {'learning_rate': 0.008471801418819975, 'n_hidden': 188}. Best is trial 1 with value: 0.8748000264167786.


...

[I 2026-09-26 18:47:19,491] Trial 2 finished with value: 0.1080000028014183 and parameters: {'learning_rate': 4.207988669606632e-05, 'n_hidden': 63}. Best is trial 1 with value: 0.8748000264167786.


Epoch 1/20, Loss: 2.2992
Training accuracy: 0.1233
Validation accuracy: 0.1176
Epoch 2/20, Loss: 2.2963
Training accuracy: 0.1316
Validation accuracy: 0.1244
Epoch 3/20, Loss: 2.2934
Training accuracy: 0.1401
Validation accuracy: 0.1326
Epoch 4/20, Loss: 2.2905
Training accuracy: 0.1497
Validation accuracy: 0.1444
Epoch 5/20, Loss: 2.2876
Training accuracy: 0.1608
Validation accuracy: 0.1548
Epoch 6/20, Loss: 2.2848
Training accuracy: 0.1712
Validation accuracy: 0.1656
Epoch 7/20, Loss: 2.2819
Training accuracy: 0.1813
Validation accuracy: 0.1762
Epoch 8/20, Loss: 2.2791
Training accuracy: 0.1934
Validation accuracy: 0.1884
Epoch 9/20, Loss: 2.2762
Training accuracy: 0.2079
Validation accuracy: 0.2022
Epoch 10/20, Loss: 2.2733
Training accuracy: 0.2239
Validation accuracy: 0.2196
Epoch 11/20, Loss: 2.2704
Training accuracy: 0.2401
Validation accuracy: 0.2340
Epoch 12/20, Loss: 2.2675
Training accuracy: 0.2542
Validation accuracy: 0.2508
Epoch 13/20, Loss: 2.2646
Training accuracy: 0.26

[I 2026-09-26 18:50:11,868] Trial 3 finished with value: 0.2939999997615814 and parameters: {'learning_rate': 1.7073967431528103e-05, 'n_hidden': 263}. Best is trial 1 with value: 0.8748000264167786.


Epoch 1/20, Loss: 1.7941
Training accuracy: 0.6364
Validation accuracy: 0.6232
Epoch 2/20, Loss: 0.9452
Training accuracy: 0.7100
Validation accuracy: 0.7058
Epoch 3/20, Loss: 0.7468
Training accuracy: 0.7522
Validation accuracy: 0.7458
Epoch 4/20, Loss: 0.6640
Training accuracy: 0.7805
Validation accuracy: 0.7688
Epoch 5/20, Loss: 0.6096
Training accuracy: 0.8014
Validation accuracy: 0.7864
Epoch 6/20, Loss: 0.5701
Training accuracy: 0.8080
Validation accuracy: 0.8006
Epoch 7/20, Loss: 0.5403
Training accuracy: 0.8186
Validation accuracy: 0.8078
Epoch 8/20, Loss: 0.5180
Training accuracy: 0.8241
Validation accuracy: 0.8112
Epoch 9/20, Loss: 0.5005
Training accuracy: 0.8321
Validation accuracy: 0.8212
Epoch 10/20, Loss: 0.4863
Training accuracy: 0.8310
Validation accuracy: 0.8218
Epoch 11/20, Loss: 0.4746
Training accuracy: 0.8354
Validation accuracy: 0.8254
Epoch 12/20, Loss: 0.4653
Training accuracy: 0.8390
Validation accuracy: 0.8266
Epoch 13/20, Loss: 0.4568
Training accuracy: 0.84

[I 2026-09-26 18:52:50,931] Trial 4 finished with value: 0.8339999914169312 and parameters: {'learning_rate': 0.002537815508265664, 'n_hidden': 218}. Best is trial 1 with value: 0.8748000264167786.


check best hyperparms it found and what their validation accuracy was 

In [21]:
study.best_params

{'learning_rate': 0.008471801418819975, 'n_hidden': 188}

In [22]:
study.best_value

0.8748000264167786

better to pass train/validation set to objective function as arguments

In [ ]:
def objective(trial, train_loader, valid_loader):
    [...]  # the rest of the function remains the same as above

# need lambda cuz optuna calls w/ just trial argument, and we pass data ourselves
objective_with_data = lambda trial: objective(
    trial, train_loader=train_loader, valid_loader=valid_loader)
study.optimize(objective_with_data, n_trials=5)

we can also prune the trial if its not performing well, skipped for now tho
- trial can report its accuracy on epoch
- optuna can decided if it should be pruned
- then prunes it 

## Write Custom Dense Module

In [24]:
class Dense(nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()
        self.mlp = nn.Sequential(
            # only output layer, no real hidden layer
            nn.Linear(n_input, n_output),
            nn.ReLU()
            )
    def forward(self, X):
        return self.mlp(X)

class DenseV2(nn.Module):
    def __init__(self, n_inputs, n_outputs):
        super().__init__()
        # save the # of inputs and outputs 
        # so the forward pass knows
        self.n_inputs = n_inputs
        self.n_outputs = n_outputs

        # randomly init weights matching shape of input/output (assign a weight to each input feature)
        #   and assign each neuron (# of outputs) a vector of weights
        # wrap it in a param so its a param of the module, and autograd tracks it
        self.weights = torch.nn.Parameter(torch.randn(n_outputs, n_inputs))

        # same for biases, but we only need 1 bias per neuron
        self.biases = torch.nn.Parameter(torch.randn(n_outputs))

        # now weights and biases are parameters of the module

    def forward(self, X):
        # so we get a matrix of input features of size (whatever, n_inputs)
        # matmul w/ the weights works, cuz (whatever, n_inputs) x (n_outputs,  n_inputs)^T (transpose to make dims match)
        # gives us (whatever, n_outputs) that is whatever rows (size of X) and n_ouputs columns (our pred(s))

        # perform the matmul and add bias step
        y_pred = X @ self.weights.T + self.biases

        # now we need to apply or activation function (ReLU)

        # relu works elementwise, so it returns the tensor w/ relu applied on every single element in it 
        y_pred = F.relu(y_pred)

        return y_pred

In [ ]:
x = torch.tensor(1.2, requires_grad=True)
y = torch.tensor(3.4, requires_grad=True)
f = torch.sin(x**2 * y)
f.backward() # compute gradients
print(x.grad, y.grad)

tensor(1.4899) tensor(0.2629)
